In [ ]:
# Scenario: AI Research Assistant for a Corporate Innovation Team
# Imagine you’re part of a corporate innovation lab that constantly reviews new AI research papers to stay ahead of
# trends. The team struggles with long PDFs full of technical jargon, and they want a quick way to ask natural questions
# about the papers instead of reading them cover to cover.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a research paper (e.g., ai_research.pdf).
# - Chunking: The chatbot splits the paper into manageable sections so no detail is lost.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, making the paper searchable by meaning rather than keywords.
# - Retriever: When someone asks, “What does this paper say about reinforcement learning?”, the retriever pulls the most relevant chunks.
# - LLM Response: The Hugging Face model (Flan-T5) generates a concise, human-readable answer using those chunks as context.
# - Chat Loop: The team can keep asking questions interactively, like a research assistant that knows the paper inside out.



# ==========================================================
# SIMPLE RAG CHATBOT (NO LANGCHAIN) — FULLY ANNOTATED
# ==========================================================

# ----------------------------------------------------------
# STEP 0 — Install Required Libraries
# ----------------------------------------------------------
# chromadb → vector database
# sentence-transformers → embedding model
# pypdf → reading PDF files
# transformers → running the LLM

# !pip install chromadb sentence-transformers pypdf transformers


# ----------------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------------

import os

# Library for reading PDF documents
from pypdf import PdfReader

# Embedding model
from sentence_transformers import SentenceTransformer

# Vector database
import chromadb

# HuggingFace model pipeline
from transformers import pipeline


# ----------------------------------------------------------
# STEP 2 — Load the PDF Document
# ----------------------------------------------------------
# This document acts as the knowledge source for RAG

print("Loading PDF document...")

reader = PdfReader("ai_research.pdf")

text = ""

# Extract text from every page
for page in reader.pages:
    text += page.extract_text()

print("Document Loaded")
print("Total Characters:", len(text))

# Preview some text
print("\nPreview:\n")
print(text[:500])


# ----------------------------------------------------------
# STEP 3 — Chunk the Document
# ----------------------------------------------------------
# LLMs work better with smaller text segments
# so we split the document into chunks

print("\nSplitting document into chunks...")

def chunk_text(text, chunk_size=500, overlap=50):
    """
    Split text into overlapping chunks

    chunk_size = max characters per chunk
    overlap = shared characters between chunks
    """

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


chunks = chunk_text(text)

print("Total Chunks Created:", len(chunks))

print("\nExample Chunk:\n")
print(chunks[0])


# ----------------------------------------------------------
# STEP 4 — Create Embeddings
# ----------------------------------------------------------
# Convert each chunk into a numerical vector
# These vectors allow semantic similarity search

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ----------------------------------------------------------
# STEP 5 — Create Vector Database
# ----------------------------------------------------------
# ChromaDB stores embeddings and documents

client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection("pdf_collection")
    print("Old collection deleted")
except:
    pass


collection = client.create_collection("pdf_collection")

print("New vector collection created")


# ----------------------------------------------------------
# STEP 6 — Store Chunks in Vector DB
# ----------------------------------------------------------

print("\nCreating embeddings and storing in ChromaDB...")

for i, chunk in enumerate(chunks):

    # Create embedding vector
    embedding = embedding_model.encode(chunk).tolist()

    # Store in vector database
    collection.add(
        documents=[chunk],
        embeddings=[embedding],
        ids=[str(i)]
    )

print("All chunks stored successfully")


# ----------------------------------------------------------
# STEP 7 — Retriever Function
# ----------------------------------------------------------
# Converts user question → embedding
# Finds most similar document chunks

def retrieve(query, k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode(query).tolist()

    # Perform similarity search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]


# ----------------------------------------------------------
# STEP 8 — Load the LLM
# ----------------------------------------------------------
# We use Flan-T5 for answer generation

print("\nLoading LLM...")

qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# ----------------------------------------------------------
# STEP 9 — Question Answering Function
# ----------------------------------------------------------
# RAG Workflow:
#
# Question
#   ↓
# Retrieve Relevant Chunks
#   ↓
# Send Context + Question to LLM
#   ↓
# Generate Answer

def answer_question(query):

    # Retrieve relevant context
    context_docs = retrieve(query)

    context = " ".join(context_docs)

    prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
{context}

Question:
{query}

Answer:
"""

    response = qa_pipeline(
        prompt,
        max_length=256,
        temperature=0.5
    )

    return response[0]["generated_text"]


# ----------------------------------------------------------
# STEP 10 — Chat Loop
# ----------------------------------------------------------
# Interactive question-answering interface

print("\n==============================")
print("RAG Chatbot Ready")
print("Type 'exit' to stop")
print("==============================\n")

while True:

    question = input("Ask a question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:\n", answer)
    print("\n" + "-"*60 + "\n")

Loading PDF document...
Document Loaded
Total Characters: 2812

Preview:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applications include healthcare
diagnostics, recommendation systems, autonomous vehicles, and conversational agents.
Machine Learning
Machine Learning is a sub

Splitting document into chunks...
Total Chunks Created: 7

Example Chunk:

Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI applicat

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Old collection deleted
New vector collection created

Creating embeddings and storing in ChromaDB...
All chunks stored successfully

Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully

RAG Chatbot Ready
Type 'exit' to stop



Both `max_new_tokens` (=256) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
 
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not present, say "Not found in document".

Context:
agents.
Machine Learning
Machine Learning is a subset of artificial intelligence that focuses on algorithms that improve
automatically through experience. Instead of being explicitly programmed, machine learning models
learn patterns from data. Common types of machine learning include supervised learning,
unsupervised learning, and reinforcement learning.
Deep Learning
Deep learning is a specialized branch of machine learning that uses neural networks with multiple
layers to model complex patter Artificial Intelligence Research Overview
Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are
programmed to think, learn, and make decisions. AI systems can analyze large datasets, recognize
patterns, and generate predictions or recommendations. Modern AI ap

In [ ]:
# Scenario: Legal Research Assistant for a Corporate Compliance Team
# Context
# A corporate compliance department constantly reviews lengthy legal documents, regulatory filings, and policy updates. These documents are dense, full of
# legal terminology, and often hundreds of pages long. The team struggles to quickly extract relevant clauses or understand implications without spending hours reading.
# How the RAG Chatbot Fits In
# - Input Source: The team uploads a legal document (e.g., data_privacy_regulation.pdf).
# - Chunking: The chatbot splits the document into sections (clauses, articles, sub-sections) so no detail is overlooked.
# - Embeddings + Vector DB: Each section is converted into embeddings and stored in Chroma, enabling semantic search rather than keyword-only lookup.
# - Retriever: When someone asks, “What does this regulation say about cross-border data transfers?”, the retriever surfaces the most relevant clauses.
# - LLM Response: A Hugging Face model (e.g., Flan-T5) generates a concise, plain-language summary of those clauses, stripping away heavy legal jargon.
# - Chat Loop: The compliance team can continue asking questions interactively, like “Does this regulation conflict with GDPR?” or “What penalties are mentioned
#  for non-compliance?”.
# Outcome
# The chatbot acts as a legal research assistant, helping the compliance team quickly interpret complex documents, identify risks, and prepare summaries for executives
#  without needing to manually parse every page.


In [ ]:
# Scenario: University Library Assistant
# A large university library has thousands of digitized textbooks, research papers, and course notes. Students often struggle to find specific explanations or summaries when preparing for exams. Instead of manually searching through PDFs, the library deploys a RAG chatbot that acts like a study companion.
# How It Works
# - Input Source: Students upload or access a textbook PDF (e.g., Introduction_to_Data_Science.pdf).
# - Chunking: The chatbot splits the textbook into smaller sections so that each concept is searchable.
# - Embeddings + Vector DB: Each section is embedded and stored in Chroma, making the textbook searchable by meaning.
# - Retriever: When a student asks, “Explain the difference between supervised and unsupervised learning,” the retriever pulls the most relevant sections.
# - LLM Response: The Hugging Face model generates a clear, concise answer tailored to the student’s query.
# - Interactive Chat: Students can keep asking follow-up questions, turning the textbook into a conversational tutor.


In [ ]:
# ==============================================================
# RAG LEGAL COMPLIANCE ASSISTANT
# GRADIO + CHROMA + EMBEDDINGS + LLM
# ==============================================================

# --------------------------------------------------------------
# STEP 1 — Install Dependencies
# --------------------------------------------------------------

# !pip install gradio chromadb sentence-transformers pypdf transformers


# --------------------------------------------------------------
# STEP 2 — Import Libraries
# --------------------------------------------------------------

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline


# --------------------------------------------------------------
# STEP 3 — Load Embedding Model
# --------------------------------------------------------------
# Converts text into vector embeddings for semantic search

print("Loading embedding model...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# --------------------------------------------------------------
# STEP 4 — Initialize Vector Database (Chroma)
# --------------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("legal_docs")
except:
    pass

collection = client.create_collection("legal_docs")


# --------------------------------------------------------------
# STEP 5 — Load Language Model
# --------------------------------------------------------------

print("Loading LLM...")

llm = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("LLM loaded successfully")


# --------------------------------------------------------------
# STEP 6 — Document Chunking
# --------------------------------------------------------------

def chunk_text(text, chunk_size=500, overlap=50):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# --------------------------------------------------------------
# STEP 7 — Process Uploaded PDF
# --------------------------------------------------------------

def process_pdf(file):

    print("Processing PDF...")

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:
        text += page.extract_text()

    chunks = chunk_text(text)

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Document processed successfully. {len(chunks)} chunks stored."


# --------------------------------------------------------------
# STEP 8 — Retriever
# --------------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    docs = results["documents"][0]

    print("\nRetrieved Chunks:\n", docs)

    return docs


# --------------------------------------------------------------
# STEP 9 — Answer Generation
# --------------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a legal compliance assistant.

Use ONLY the context below to answer the question.

Context:
{context}

Question: {query}

Provide a short clear answer.
"""

    response = llm(
        prompt,
        max_length=200,
        temperature=0.2
    )

    result = response[0]["generated_text"]

    print("\nRaw Model Output:\n", result)

    return result


# --------------------------------------------------------------
# STEP 10 — Chat Function
# --------------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a question."

    answer = answer_question(question)

    if not answer:
        return "No answer generated."

    return answer


# --------------------------------------------------------------
# STEP 11 — Build Gradio Interface
# --------------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 📜 Legal Compliance RAG Assistant")

    gr.Markdown("""
Upload a legal regulation document and ask questions about:

• compliance rules
• cross-border data transfers
• penalties for violations
• data subject rights
""")

    pdf_input = gr.File(label="Upload Legal PDF")

    upload_button = gr.Button("Process Document")

    status = gr.Textbox(label="Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Legal Question"
    )

    answer_box = gr.Textbox(
        label="Answer",
        lines=15
    )

    ask_button = gr.Button("Ask")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )


# --------------------------------------------------------------
# STEP 12 — Launch Application
# --------------------------------------------------------------

demo.launch()

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Loading LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

LLM loaded successfully
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://74166d71d47b83f667.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Scenario: Healthcare Clinical Research Assistant
Context

Hospitals and medical research teams regularly analyze large medical documents such as:

clinical trial reports

treatment guidelines

drug safety reports

patient care protocols

These documents are very long and technical, often containing:

medical terminology

treatment procedures

dosage guidelines

adverse drug reactions

Doctors and healthcare researchers often spend hours searching for specific information, such as:

recommended drug dosages

treatment protocols for specific diseases

safety warnings and contraindications